# 02 — Feature Engineering

**Pipeline stage 2 of 5**

## Objective
Build the feature set used by every downstream model, parameterized so the
*same* feature function serves all three forecast tiers (7–14, 30, 60–90 day).

**This is exploratory only — it is not what's live.** The code below has
since been extracted into `quant/features.py`, which is the canonical
implementation; `scripts/build_quant_features.py` runs it in production
with no notebook execution required. This notebook is *not* related to
`scripts/train_models.py::build_features` (a previous version of this cell
claimed it mirrored that function — it never did; that function builds a
different, single flat feature set for the separate point-prediction model
loaded by `backend/app/model.py`). If you re-run this notebook, prefer
importing from `quant.features` over re-defining these functions inline,
so the two can't drift apart again.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "prices_clean.parquet")
df = df.sort_values(["commodity", "market", "date"]).reset_index(drop=True)
df.shape

## 1. Temporal features

Cyclic month encoding avoids the false discontinuity a raw month integer creates (December=12, January=1 look far apart numerically but are adjacent seasonally).

In [ ]:
def add_temporal_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["year"] = frame["date"].dt.year
    frame["month"] = frame["date"].dt.month
    frame["quarter"] = frame["date"].dt.quarter
    frame["week"] = frame["date"].dt.isocalendar().week.astype(int)
    frame["day_of_year"] = frame["date"].dt.dayofyear
    frame["month_sin"] = np.sin(2 * np.pi * frame["month"] / 12)
    frame["month_cos"] = np.cos(2 * np.pi * frame["month"] / 12)
    return frame

df = add_temporal_features(df)
df[["date", "month", "month_sin", "month_cos"]].head()

## 2. Lag & rolling features, per tier horizon

The forecasting horizon determines which lags are informative. A 7-day tier benefits from a short lag (1 observation back); a 90-day tier needs longer lags so the model isn't extrapolating past its own recent noise. `lag_steps` is expressed in **observations**, not days, since observation frequency varies by crop×market (see notebook 01's coverage report).

In [ ]:
def add_lag_features(frame: pd.DataFrame, lag_steps=(1, 3, 6), roll_window: int = 3) -> pd.DataFrame:
    frame = frame.copy()
    group_cols = ["commodity", "market"]
    for step in lag_steps:
        frame[f"price_lag_{step}"] = frame.groupby(group_cols)["price"].shift(step)
    frame[f"price_roll_{roll_window}_avg"] = (
        frame.groupby(group_cols)["price"]
        .transform(lambda s: s.rolling(roll_window, min_periods=1).mean())
    )
    frame[f"price_roll_{roll_window}_std"] = (
        frame.groupby(group_cols)["price"]
        .transform(lambda s: s.rolling(roll_window, min_periods=2).std())
    )
    return frame

# Tier-specific lag sets: near-term tier uses short lags, directional tier uses longer lags
TIER_LAG_CONFIG = {
    "tier_7_14":  {"lag_steps": (1, 2, 3),  "roll_window": 2},
    "tier_30":    {"lag_steps": (1, 3, 6),  "roll_window": 3},
    "tier_60_90": {"lag_steps": (2, 6, 12), "roll_window": 6},
}

tier_frames = {
    tier: add_lag_features(df, **cfg)
    for tier, cfg in TIER_LAG_CONFIG.items()
}
tier_frames["tier_30"].head()

## 3. Categorical encoding

Label encoding, not one-hot — consistent with production (XGBoost handles ordinal-encoded categoricals well and this keeps the feature matrix small enough for low-resource deployment, which matters given the fail-safe/offline design goal).

In [ ]:
le_crop = LabelEncoder()
le_market = LabelEncoder()
le_crop.fit(df["commodity"].astype(str))
le_market.fit(df["market"].astype(str))

FEATURE_COLS_BASE = [
    "crop_enc", "market_enc",
    "year", "month", "quarter", "week", "day_of_year",
    "month_sin", "month_cos",
]

def finalize_features(frame: pd.DataFrame, lag_steps, roll_window) -> tuple[pd.DataFrame, pd.Series, list]:
    frame = frame.copy()
    frame["crop_enc"] = le_crop.transform(frame["commodity"].astype(str))
    frame["market_enc"] = le_market.transform(frame["market"].astype(str))
    lag_cols = [f"price_lag_{s}" for s in lag_steps] + [
        f"price_roll_{roll_window}_avg", f"price_roll_{roll_window}_std",
    ]
    feature_cols = FEATURE_COLS_BASE + lag_cols
    frame = frame.dropna(subset=feature_cols + ["price"])
    return frame[feature_cols], frame["price"], feature_cols

datasets = {}
for tier, cfg in TIER_LAG_CONFIG.items():
    X, y, feature_cols = finalize_features(tier_frames[tier], **cfg)
    datasets[tier] = {"X": X, "y": y, "feature_cols": feature_cols, "dates": tier_frames[tier].loc[X.index, "date"]}
    print(f"{tier:12s} → {len(X):,} rows, {len(feature_cols)} features")

## 4. Persist feature sets

Each tier's feature matrix is saved separately — notebook 03 loads these directly rather than recomputing, keeping the backtest reproducible and fast to re-run.

In [ ]:
import joblib

for tier, data in datasets.items():
    data["X"].assign(price=data["y"], date=data["dates"]).to_parquet(
        PROCESSED_DIR / f"features_{tier}.parquet", index=False
    )

joblib.dump(
    {"le_crop": le_crop, "le_market": le_market, "tier_lag_config": TIER_LAG_CONFIG},
    PROCESSED_DIR / "feature_encoders.pkl",
)
print("Saved feature sets for:", list(datasets.keys()))

## Output

- `data/processed/features_tier_7_14.parquet`, `features_tier_30.parquet`, `features_tier_60_90.parquet`
- `data/processed/feature_encoders.pkl` — shared label encoders + tier lag config, reused at inference time so training and serving never drift apart

**Next:** `03_walkforward_backtesting.ipynb`